<a href="https://www.kaggle.com/code/fredericnicholson/pyspark-logistic-regression-half-dataset?scriptVersionId=238811294" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
# keep the submission time low 
training = True 

import numpy as np # linear algebra


import os

if training :
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

        

/kaggle/input/playground-series-s4e7/sample_submission.csv
/kaggle/input/playground-series-s4e7/train.csv
/kaggle/input/playground-series-s4e7/test.csv


In [3]:
! pip install  pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=a272a3e1707d3871726569ed403a822c8252ef317cc9e4f8329a34c8aabf4bf9
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [4]:
from pyspark.sql import SparkSession
# instead of native pandas we will use the pyspark version 
import pyspark.pandas as ps

# local mean kaggle cloud 
spark = SparkSession.builder.master('local').getOrCreate()

spark

/opt/conda/lib/python3.10/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/13 22:31:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# reading the training dataset. Note that schema can be replaced by other options  
train = spark.read.csv('/kaggle/input/playground-series-s4e7/train.csv', 
                       schema ='id INT, Gender STRING, Age INT, Driving_License INT, Region_Code FLOAT, Previously_Insured INT, Vehicle_Age STRING, Vehicle_Damage STRING, Annual_Premium FLOAT, Policy_Sales_Channel FLOAT, Vintage FLOAT, Response INT', header=True)
print (train.schema)

# this can keep the dataset small 
train = train.sample(fraction=1.0, seed=42)  

# only get data for nummberic values 
if training :
    print (train.select("Age", "Driving_License", "Region_Code", "Previously_Insured" ).describe().show())
    print (train.select("Annual_Premium","Policy_Sales_Channel","Vintage","Response").describe().show())

#for c in train.columns :
    
#    col = train [c]
#    print (c, " type : ", type (col))
    # print (col.distinct())
    

StructType([StructField('id', IntegerType(), True), StructField('Gender', StringType(), True), StructField('Age', IntegerType(), True), StructField('Driving_License', IntegerType(), True), StructField('Region_Code', FloatType(), True), StructField('Previously_Insured', IntegerType(), True), StructField('Vehicle_Age', StringType(), True), StructField('Vehicle_Damage', StringType(), True), StructField('Annual_Premium', FloatType(), True), StructField('Policy_Sales_Channel', FloatType(), True), StructField('Vintage', FloatType(), True), StructField('Response', IntegerType(), True)])


24/07/13 22:31:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+-------------------+------------------+-------------------+
|summary|              Age|    Driving_License|       Region_Code| Previously_Insured|
+-------+-----------------+-------------------+------------------+-------------------+
|  count|         11504798|           11504798|          11504798|           11504798|
|   mean|38.38356336199905| 0.9980219557092614|26.418689767521407|0.46299656890977137|
| stddev|14.99345850838089|0.04443120303474215|12.991590202215061|0.49862888774978975|
|    min|               20|                  0|               0.0|                  0|
|    max|               85|                  1|              52.0|                  1|
+-------+-----------------+-------------------+------------------+-------------------+

None


+-------+------------------+--------------------+------------------+-------------------+
|summary|    Annual_Premium|Policy_Sales_Channel|           Vintage|           Response|
+-------+------------------+--------------------+------------------+-------------------+
|  count|          11504798|            11504798|          11504798|           11504798|
|   mean|30461.370410588694|  112.42544188954903|163.89774388042275|0.12299729208631043|
| stddev|16454.745205061354|  54.035707776861486| 79.97953110341078|0.32843411455227944|
|    min|            2630.0|                 1.0|              10.0|                  0|
|    max|          540165.0|               163.0|             299.0|                  1|
+-------+------------------+--------------------+------------------+-------------------+

None


In [5]:
if training :
    print (train.groupBy (['Vehicle_age', 'Response']).count().sort(['Vehicle_age']).show())
    print (train.groupBy (['Region_code', 'Response']).count().sort (['Region_code']).show(500))

+-----------+--------+-------+
|Vehicle_age|Response|  count|
+-----------+--------+-------+
|   1-2 Year|       1|1063272|
|   1-2 Year|       0|4919406|
|   < 1 Year|       0|4835296|
|   < 1 Year|       1| 208849|
|  > 2 Years|       1| 142938|
|  > 2 Years|       0| 335037|
+-----------+--------+-------+

None


+-----------+--------+-------+
|Region_code|Response|  count|
+-----------+--------+-------+
|        0.0|       0|  54867|
|        0.0|       1|   4407|
|        1.0|       0|  30499|
|        1.0|       1|   3467|
|        2.0|       1|   8467|
|        2.0|       0| 109630|
|        3.0|       1|  31299|
|        3.0|       0| 215004|
|        4.0|       0|  44197|
|        4.0|       1|   8307|
|        5.0|       1|   4145|
|        5.0|       0|  32687|
|        6.0|       1|  12328|
|        6.0|       0| 168794|
|        7.0|       1|  11710|
|        7.0|       0|  80530|
|        8.0|       0| 931473|
|        8.0|       1|  89563|
|        9.0|       1|   7517|
|        9.0|       0|  85854|
|       10.0|       0| 118713|
|       10.0|       1|   7368|
|       11.0|       0| 248268|
|       11.0|       1|  29993|
|       12.0|       0|  82864|
|       12.0|       1|   9278|
|       13.0|       0|  99158|
|       13.0|       1|   9680|
|       14.0|       1|  11534|
|       

In [7]:
# removing outlyer 
train.filter((train ['Region_code'] > 39.2) & (train ['Region_code'] < 39.3)).show()
train = train.filter(train ['id'] !=  11370234) 

+--------+------+---+---------------+-----------+------------------+-----------+--------------+--------------+--------------------+-------+--------+
|      id|Gender|Age|Driving_License|Region_Code|Previously_Insured|Vehicle_Age|Vehicle_Damage|Annual_Premium|Policy_Sales_Channel|Vintage|Response|
+--------+------+---+---------------+-----------+------------------+-----------+--------------+--------------+--------------------+-------+--------+
|11370234|Female| 20|              1|       39.2|                 1|   < 1 Year|            No|        2630.0|               159.0|   74.0|       0|
+--------+------+---+---------------+-----------+------------------+-----------+--------------+--------------+--------------------+-------+--------+



creating a more balanced DataFrame. This will reduce the size of the data to be processed and also improve the learn capabilty.   

In [8]:
train.groupby('Response').count().show()

+--------+--------+
|Response|   count|
+--------+--------+
|       1| 1415059|
|       0|10089738|
+--------+--------+



In [9]:
%%time
pos_size   = 1415059
neg_size = 10089738
fract = pos_size/(pos_size + neg_size)

positive_df = train [train ['Response'] == 1]
negative_df = train [train ['Response'] == 0]
negative_df_balanced = negative_df.sample (fraction = fract, seed = 42 )


CPU times: user 1.06 ms, sys: 4.74 ms, total: 5.8 ms
Wall time: 20.7 ms


In [10]:
print (type (positive_df))
print (type (negative_df_balanced))

balanced_train = ps.concat ([ps.DataFrame (positive_df), ps.DataFrame (negative_df_balanced) ])

type (balanced_train)

<class 'pyspark.sql.dataframe.DataFrame'>
<class 'pyspark.sql.dataframe.DataFrame'>


24/07/13 22:45:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


pyspark.pandas.frame.DataFrame

In [13]:
string_features = [ "Gender",  "Vehicle_Age", "Vehicle_Damage"] 
#Gender and damage are binary, so no need to treat them as category 
cat_features = [ "Vehicle_Age_idx", "Region_Code", "Policy_Sales_Channel"]
binary_features = ['Gender_idx', 'Driving_License', 'Previously_Insured', 'Vehicle_Damage_idx'] 
num_features = ['Age',  'Annual_Premium', 'Vintage']

we are building a pipeline with the pyspark options

this is a list of all the features 

 ['id', 'Gender', 'Age', 'Driving_License', 'REGION_Code', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage', 
 'Annual_Premium', 'Policy_Sales_Channel', 'Vintage', 'Response']



In [14]:

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# one string to number conversion per feature
string_indexer = [ StringIndexer(inputCol=f, outputCol= f + "_idx") for f in string_features] 

# one one hote encoder for all features 
one_hot_encoder_output_cols = [ c + "_one_hot_encoder" for c in  cat_features] 
one_hot_encoder = OneHotEncoder (inputCols = cat_features, outputCols =  one_hot_encoder_output_cols)

# one Vectur assember to convert the data suitable for pySpark machine learning library.   All features must be in a packed in 
# one column namaed 'features'. the actual features are in vectors. This is done in the foillowing step   
assembler_input = binary_features + num_features + one_hot_encoder_output_cols

vector_assembler = VectorAssembler (inputCols = assembler_input, outputCol = 'vector_of_features')

stages = []

stages += string_indexer
stages += [one_hot_encoder]
stages += [vector_assembler]

stages

[StringIndexer_4062ffc36e3c,
 StringIndexer_ae7184d065c8,
 StringIndexer_614968745c7b,
 OneHotEncoder_249b2e2b2d8e,
 VectorAssembler_a22c9c4cde80]

In [16]:
%%time
from pyspark.ml import Pipeline

pipeline = Pipeline ().setStages (stages)

print (f'type = {type (balanced_train)},  shape before :   {balanced_train.shape}')

pipe_input = spark.createDataFrame(balanced_train)

pipe_model = pipeline.fit (pipe_input)

print ('fitting completed')

train_piped = pipe_model.transform (pipe_input)

print (f'shape after :  {train_piped.shape}')


type = <class 'pyspark.pandas.frame.DataFrame'>,  shape before :   (2656819, 12)


PySparkTypeError: [CANNOT_INFER_SCHEMA_FOR_TYPE] Can not infer schema for type: `str`.

In [22]:
%%time 
from pyspark.sql.functions import col 

# need to rename the columns to match pySpark conventions 
train_data = train_piped.select (col ('vector_of_features').alias ('features'), 
                                 col ('Response').alias ('label'))

CPU times: user 4.21 ms, sys: 821 µs, total: 5.03 ms
Wall time: 16.4 ms


In [23]:
%%time 
train_data.groupBy ('label').count ().show()

+-----+--------+
|label|   count|
+-----+--------+
|    1| 1415059|
|    0|10089738|
+-----+--------+

CPU times: user 19 ms, sys: 1.02 ms, total: 20 ms
Wall time: 19.2 s


In [14]:
%%time 

from pyspark.ml.classification import  LogisticRegression
from pyspark.ml.classification import MultilayerPerceptronClassifier
# Build the model


layers = [220, 32, 32, 2]
trainer = MultilayerPerceptronClassifier(maxIter=50, layers=layers, blockSize=128, seed=1234)

# train the model
model = trainer.fit(train_data)
# model = LogisticRegression().fit(dataset = train_data)
# model = ().fit(dataset = train_data)

# Evaluating the model on training data



NameError: name 'train_data' is not defined

In [15]:
if training :
    model.summary.areaUnderROC

NameError: name 'model' is not defined

In [ ]:
if training :
    model.summary.pr.show()

In [ ]:
%%time 
submission_data  = spark.read.csv('/kaggle/input/playground-series-s4e7/test.csv', 
                       schema ='id INT, Gender STRING, Age INT, Driving_License INT, Region_Code FLOAT, Previously_Insured INT, Vehicle_Age STRING, Vehicle_Damage STRING, Annual_Premium FLOAT, Policy_Sales_Channel FLOAT, Vintage FLOAT', header=True)
 
submission_piped = pipe_model.transform (submission_data)
    
print ('finished')



In [ ]:
%%time 
predict = model.transform (submission_piped.select (col ('vector_of_features').alias ('features'), col ('id')))

print (predict.schema)
print ('prediction finished')

In [ ]:
probability = predict.select (col ('probability'), col ('id'))

# prob_DF = probability.toDF('Response', 'id')

In [ ]:
%%time 
# this code will extract the second value from the probability vector, as this value is the probability for a positive response.  

from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

second_element = udf(lambda v:float(v[1]),FloatType())
submission = probability.select(col ('id'), second_element('probability')).toPandas()
submission = submission.rename (columns = { '<lambda>(probability)' : 'Response'})
submission.head ()

In [ ]:
submission.to_csv("submission.csv", index = False)